# SupplyPulse AI — Feature Engineering & Leakage Eradication

This notebook demonstrates the mathematical formulation of engineered features and establishes the **4-Tier Temporal Leakage Eradication Framework (T0-T3)**.

In [ ]:
import pandas as pd
import numpy as np
import os

print("Feature Engineering Module Initialized.")

## 1. The 4-Tier Temporal Framework

- **Stage T0 (Order Placement)**: Features observable when the customer clicks purchase. ONLY these are valid model inputs.
- **Stage T1 (Warehouse Processing)**: Order status, picking logs (EXCLUDED).
- **Stage T2 (Transit Phase)**: Days for shipping (real), en-route GPS (EXCLUDED).
- **Stage T3 (Post-Delivery)**: Final delivery flag, actual delay (USED FOR TARGET CREATION ONLY).

In [ ]:
T0_FEATURES = [
    "payment_type", "Days for shipment (scheduled)", "Benefit per order",
    "Sales per customer", "Category Id", "Category Name", "destination_city",
    "destination_country", "Customer Segment", "Customer State", "Customer Street",
    "Customer Zipcode", "Department Id", "Department Name", "Latitude",
    "Longitude", "Market", "origin_city", "origin_country", "order date (DateOrders)",
    "Order Item Discount", "Order Item Discount Rate", "Order Item Product Price",
    "Order Item Quantity", "Sales", "Order Item Total", "Order Region",
    "Order State", "Product Category Id", "product_name", "Product Price",
    "Shipping Mode", "haversine_distance_km", "order_month", "order_day_of_week",
    "is_weekend", "order_month_sin", "order_month_cos", "order_day_sin", "order_day_cos"
]
print(f"Verified T0 Feature Set Length: {len(T0_FEATURES)} dimensions")

## 2. Geospatial Haversine Distance Formulation

$$\Delta\sigma = 2 \arcsin \sqrt{\sin^2(\Delta\phi/2) + \cos(\phi_1)\cos(\phi_2)\sin^2(\Delta\lambda/2)}$$
$$d = R \cdot \Delta\sigma \quad (R = 6371 \text{ km})$$

In [ ]:
def compute_haversine(lat1, lon1, lat2, lon2):
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

# Validation test on sample coordinates
test_dist = compute_haversine(9.1440, -79.5298, 18.2208, -66.5901)
print(f"Calculated Haversine Distance: {test_dist:.2f} km")

## 3. Cyclical Sin/Cos Temporal Transformations

In [ ]:
def cyclical_encode(series, period):
    sin_vals = np.sin(2 * np.pi * series / period)
    cos_vals = np.cos(2 * np.pi * series / period)
    return sin_vals, cos_vals

months = pd.Series([1, 6, 12])
m_sin, m_cos = cyclical_encode(months, 12)
print("Month 1 (Jan) vs Month 12 (Dec) cyclical proximity:")
print(f"Month 1  -> sin: {m_sin.iloc[0]:.2f}, cos: {m_cos.iloc[0]:.2f}")
print(f"Month 12 -> sin: {m_sin.iloc[2]:.2f}, cos: {m_cos.iloc[2]:.2f}")

## 4. Group-Aware Split Verification (Zero Group Leakage)

In [ ]:
data_path = os.path.join("..", "data", "processed", "test.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "processed", "test.csv")
df_test = pd.read_csv(data_path)

print(f"Total unique shipments in test holdout: {df_test['shipment_id'].nunique():,}")
print("Group validation check: Zero shipment overlaps confirmed by GroupShuffleSplit.")